In [ ]:
# --- Librerías Estándar ---
import json
import time
import warnings
from collections import Counter, defaultdict
from pathlib import Path

# --- Librerías de Terceros (General) ---
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import cm
from tqdm.auto import tqdm

# --- Machine Learning y Datos ---
import open_clip
import torch
import torch.nn as nn
import torchvision.transforms.v2 as T
from datasets import load_dataset
from safetensors.torch import load_file
from sklearn.metrics import (
    confusion_matrix, 
    f1_score, 
    precision_score, 
    recall_score
)

from scipy.ndimage import gaussian_filter1d
from openTSNE import TSNE

from sklearn.neighbors import KNeighborsClassifier
from torch.utils.data import DataLoader
from transformers import AutoModelForImageClassification

# --- Librerías Locales / Específicas ---
from planktonzilla.clip_model import *

# --- Configuración ---
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

In [ ]:
import sys
sys.path.append("/lustre/fswork/projects/rech/tec/uod68bo/am")

In [ ]:
#train_dataset = load_dataset("project-oceania/planktonzilla_only_plankton", split="train")
test_dataset = load_dataset("project-oceania/planktonzilla_only_plankton", split="test")

In [ ]:
train_dataset = load_dataset("project-oceania/planktonzilla_only_plankton", split="train")

train_labels = train_dataset["label"]
class_frequencies = dict(Counter(train_labels))

## Functions

In [ ]:
def clip_preds_and_features(
    model,
    dataset,
    tokenizer,
    preprocess,
    class_names,
    prompt_template=None,
    allowed_indices=None
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # 1. Enviar modelo a GPU primero
    model.to(device)
    
    # 2. Configurar Multi-GPU y definir helpers
    multi_gpu = False
    if torch.cuda.device_count() > 1:
        print(f"Usando {torch.cuda.device_count()} GPUs con DataParallel")
        model = nn.DataParallel(model)
        multi_gpu = True

    model.eval()

    # Helpers: Ahora model.module solo se accede si multi_gpu es True
    def encode_text(x): 
        return model.module.encode_text(x) if multi_gpu else model.encode_text(x)

    def encode_image(x): 
        return model.module.encode_image(x) if multi_gpu else model.encode_image(x)

    # 3. Procesar clases
    if prompt_template is not None:
        class_names = [prompt_template.format(label=c) for c in class_names]

    text_tok = tokenizer(class_names).to(device)
    with torch.no_grad():
        text_features = encode_text(text_tok)
        text_features /= text_features.norm(dim=-1, keepdim=True)

    # 4. Máscara de logits
    mask = None
    if allowed_indices is not None:
        mask = torch.full((len(class_names),), float('-inf')).to(device)
        mask[allowed_indices] = 0.0

    # 5. Dataloader
    def collate_fn(batch):
        images = torch.stack([preprocess(s["image"]) for s in batch])
        labels = torch.tensor([s["label"] for s in batch])
        return {"pixel_values": images, "labels": labels}

    dataloader = DataLoader(
        dataset, 
        batch_size=128, 
        shuffle=False, 
        num_workers=8, 
        collate_fn=collate_fn,
        pin_memory=True
    )

    all_labels, all_preds, all_features = [], [], []

    with torch.no_grad(), torch.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu"):
        for batch in tqdm(dataloader, desc="Inference"):
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"]

            image_features = encode_image(pixel_values)
            all_features.append(image_features.cpu())

            image_features /= image_features.norm(dim=-1, keepdim=True)
            
            logits = 100.0 * image_features @ text_features.T
            if mask is not None:
                logits = logits + mask 

            preds = logits.argmax(dim=1)

            all_labels.extend(labels.tolist())
            all_preds.extend(preds.cpu().tolist())

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_features = torch.cat(all_features, dim=0).numpy()

    
    return all_labels, all_preds, all_features

In [ ]:
def clf_preds_and_features(model, dataset, transform):
    """
    Realiza inferencia en un dataset utilizando Multi-GPU si está disponible.
    Retorna etiquetas reales, predicciones y vectores de características (features).
    """
    
    def collate_fn(batch):
        pixel_values = torch.stack([x["pixel_values"] for x in batch])
        labels = torch.tensor([x["label"] for x in batch], dtype=torch.long)
        return {"pixel_values": pixel_values, "labels": labels}

    def preprocess(batch):
        # Aplicamos la transformación a la lista de imágenes "al vuelo"
        batch["pixel_values"] = [transform(img) for img in batch["image"]]
        return batch

    # Configurar el dataset para usar la función de preprocesamiento
    dataset.set_transform(preprocess)

    # Configuración del DataLoader
    dataloader = DataLoader(
        dataset, 
        batch_size=128, 
        shuffle=False, 
        collate_fn=collate_fn,
        num_workers=8,  
        pin_memory=True 
    )

    # Detectar y configurar el dispositivo
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Manejo de Multi-GPU con DataParallel
    if torch.cuda.device_count() > 1:
        print(f"🚀 ¡Activando Turbo! Usando {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)
    
    model.to(device)
    model.eval()

    all_preds = []
    all_labels = []
    all_features = [] # Lista para acumular los tensores de características

    print(f"🧠 Iniciando inferencia en {device}...")

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Inferencia", unit="batch"):
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            # El modelo devuelve ImageClassifierOutput
            outputs = model(pixel_values)
            
            # 1. Obtener Logits y Predicciones
            logits = outputs.logits
            preds = logits.softmax(dim=1).argmax(dim=1)

            # 2. Obtener Features (vienen en hidden_states como una tupla)
            # Extraemos el primer elemento de la tupla: [Batch_Size, Num_Features]
            features = outputs.hidden_states[0]

            # Movemos a CPU y guardamos
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            all_features.append(features.cpu())

    # Concatenamos todos los batches de features en un único array de NumPy
    # Esto resulta en una matriz de forma (Total_Imagenes, Num_Features)
    final_features = torch.cat(all_features, dim=0).numpy()

    return all_labels, all_preds, final_features


In [ ]:
def plot_confusion_matrix_with_taxonomy(
    y_true,
    y_pred,
    label_names_full,
    taxonomy_levels,
    taxonomy_colors,
    normalize="true",
    cmap="Blues",
    figsize=(11, 5),
    savepath=None,
):
    """
    Parameters
    ----------
    y_true, y_pred : array-like
        Labels verdaderos y predichos (índices originales).
    label_names_full : list[str]
        Lista completa de nombres de clases (indexada por label).
    taxonomy_levels : list[str]
        Niveles taxonómicos a encerrar en orden jerárquico.
        Ej: ["kingdom", "phylum", "class", "order"]
    taxonomy_colors : list[str]
        Colores para cada nivel (mismo orden).
        Ej: ["black", "purple", "darkgreen", "orange"]
    normalize : str or None
        Normalización de confusion_matrix.
    cmap : str
        Colormap para la matriz.
    figsize : tuple
        Tamaño de la figura.
    savepath : str or None
        Ruta para guardar la figura.
    """

    assert len(taxonomy_levels) == len(taxonomy_colors), \
        "taxonomy_levels y taxonomy_colors deben tener el mismo largo"

    # ==============================
    # 1. Clases presentes en test
    # ==============================
    present_classes = np.unique(y_true)
    old_to_new = {old: new for new, old in enumerate(present_classes)}

    yt = np.array([old_to_new[y] for y in y_true])
    yp = np.array([old_to_new[y] for y in y_pred])

    label_names = [label_names_full[i] for i in present_classes]
    num_classes = len(label_names)

    # ==============================
    # 2. Matriz de confusión
    # ==============================
    cm = confusion_matrix(
        yt,
        yp,
        labels=np.arange(num_classes),
        normalize=normalize,
    )

    # ==============================
    # 3. Parsear taxonomía desde nombres
    # ==============================
    # Asume: "kingdom phylum class order ..."
    split_names = [name.split() for name in label_names]

    level_to_index = {
        level: i for i, level in enumerate(taxonomy_levels)
    }

    taxonomy_values = {
        level: [
            parts[i] if len(parts) > i else None
            for parts in split_names
        ]
        for level, i in level_to_index.items()
    }

    # ==============================
    # 4. Rangos por nivel taxonómico
    # ==============================
    taxonomy_ranges = {}

    for level in taxonomy_levels:
        value_to_indices = defaultdict(list)
        for idx, val in enumerate(taxonomy_values[level]):
            if val is not None:
                value_to_indices[val].append(idx)

        taxonomy_ranges[level] = {
            val: (min(idxs), max(idxs))
            for val, idxs in value_to_indices.items()
        }

    # ==============================
    # 5. Plot
    # ==============================
    fig, axes = plt.subplots(
        1, 2,
        figsize=figsize,
        sharex=True,
        sharey=True,
    )

    # ---- Izquierda: CM simple ----
    im0 = axes[0].imshow(cm, cmap=cmap)
    axes[0].set_title("Confusion Matrix")
    axes[0].set_xticks([])
    axes[0].set_yticks([])

    # ---- Derecha: CM taxonómica ----
    im1 = axes[1].imshow(cm, cmap=cmap)
    axes[1].set_title("Confusion Matrix (Taxonomy)")
    axes[1].set_xticks([])
    axes[1].set_yticks([])

    # Dibujar rectángulos
    zorder_base = 2
    for level, color in zip(taxonomy_levels, taxonomy_colors):
        for _, (i0, i1) in taxonomy_ranges[level].items():
            size = i1 - i0 + 1
            rect = patches.Rectangle(
                (i0 - 0.5, i0 - 0.5),
                size,
                size,
                linewidth=1,
                edgecolor=color,
                facecolor="none",
                zorder=zorder_base,
            )
            axes[1].add_patch(rect)
        zorder_base += 1

    # ==============================
    # 6. Colorbar
    # ==============================
    fig.colorbar(
        im0,
        ax=axes[1],
        fraction=0.046,
        pad=0.04,
    )

    fig.tight_layout()

    # ==============================
    # 7. Guardar
    # ==============================
    if savepath is not None:
        fig.savefig(
            savepath,
            dpi=300,
            bbox_inches="tight",
        )

    plt.show()

    macro_precision = precision_score(yt, yp, average="macro", zero_division=0)
    macro_recall = recall_score(yt, yp, average="macro", zero_division=0)
    macro_f1 = f1_score(yt, yp, average="macro", zero_division=0)
    
    print(f"Macro Precision: {macro_precision:.4f}")
    print(f"Macro Recall:    {macro_recall:.4f}")
    print(f"Macro F1:        {macro_f1:.4f}")

In [ ]:
def evaluate_taxonomic_metrics(y_true, y_pred, class_names):
    """
    Retorna precision / recall / f1 macro por nivel taxonómico.
    Maneja clases con taxonomías incompletas evaluando la ruta disponible.

    Además calcula 'outside_parent_rate': el porcentaje de errores en un 
    nivel que son consecuencia de un error en el nivel taxonómico superior.
    """

    TAXONOMIC_LEVELS = [
        "kingdom", "phylum", "class", "order", 
        "family", "genus", "species"
    ]
    
    results = {}

    for level_idx, level_name in enumerate(TAXONOMIC_LEVELS):
        y_true_bin = []
        y_pred_bin = []

        # n_incorrect = 0
        # n_outside_parent = 0

        for yt, yp in zip(y_true, y_pred):
            true_tokens = class_names[yt].split()
            pred_tokens = class_names[yp].split()

            # Tomamos la taxonomía concatenada HASTA el nivel actual.
            # Si los tokens son más cortos que el nivel, Python maneja 
            # el slice [:level_idx + 1] sin arrojar error, trayendo todo lo disponible.
            true_label = " ".join(true_tokens[:level_idx + 1])
            pred_label = " ".join(pred_tokens[:level_idx + 1])

            y_true_bin.append(true_label)
            y_pred_bin.append(pred_label)

        #     # -------- MÉTRICA: outside_parent_rate --------
        #     if pred_label != true_label:
        #         n_incorrect += 1

        #         # Comparamos el "padre" (el string concatenado hasta el nivel anterior)
        #         if level_idx > 0:
        #             true_parent = " ".join(true_tokens[:level_idx])
        #             pred_parent = " ".join(pred_tokens[:level_idx])
                    
        #             if pred_parent != true_parent:
        #                 n_outside_parent += 1

        # # Cálculo de la tasa de error por padre incorrecto
        # outside_parent_rate = (
        #     n_outside_parent / n_incorrect if n_incorrect > 0 else 0.0
        # )

        results[level_name] = {
            "precision": precision_score(
                y_true_bin, y_pred_bin, zero_division=0, average='macro'
            ),
            "recall": recall_score(
                y_true_bin, y_pred_bin, zero_division=0, average='macro'
            ),
            "f1": f1_score(
                y_true_bin, y_pred_bin, zero_division=0, average='macro'
            ),
            # "n_incorrect": n_incorrect,
            # "n_outside_parent": n_outside_parent,
            # "outside_parent_rate": outside_parent_rate,
        }

    return results

In [ ]:
def compute_macro_metrics_per_dataset(
    y_true,
    y_pred,
    dataset_names,
    labels=None,
    zero_division=0,
):
    """
    Calcula métricas macro por dataset.

    Parameters
    ----------
    y_true, y_pred : array-like
        Labels verdaderos y predichos.
    dataset_names : array-like
        Nombre del dataset de cada ejemplo (mismo largo que y_true).
    labels : array-like or None
        Lista global de labels a considerar. Si es None, se usan
        solo las clases presentes en cada dataset.
    zero_division : int
        Valor a usar cuando hay división por cero (sklearn).

    Returns
    -------
    pd.DataFrame
        Una fila por dataset con métricas macro.
    """

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    dataset_names = np.asarray(dataset_names)

    assert len(y_true) == len(y_pred) == len(dataset_names), \
        "y_true, y_pred y dataset_names deben tener el mismo largo"

    rows = []

    for dname in np.unique(dataset_names):
        mask = dataset_names == dname
        yt = y_true[mask]
        yp = y_pred[mask]

        if labels is None:
            used_labels = np.unique(yt)
        else:
            used_labels = labels

        row = {
            "dataset": dname,
            "precision_macro": precision_score(
                yt, yp, labels=used_labels,
                average="macro", zero_division=zero_division
            ),
            "recall_macro": recall_score(
                yt, yp, labels=used_labels,
                average="macro", zero_division=zero_division
            ),
            "f1_macro": f1_score(
                yt, yp, labels=used_labels,
                average="macro", zero_division=zero_division
            ),
        }

        rows.append(row)

    return pd.DataFrame(rows).sort_values("dataset").reset_index(drop=True)


In [ ]:
def plot_metrics_vs_frequency(
    y_true,
    y_pred,
    class_frequencies,
    min_samples=1,
    n_bins=20,
    use_median=False,
    smooth_sigma=1,
    figsize=(14, 10),
):
    """
    Genera:
    - Scatter de Recall y Precision vs frecuencia
    - Promedio por bins en escala log
    - Tendencia suavizada (Gaussian) sobre los bins

    Parameters
    ----------
    n_bins : int
        Número de bins en espacio log10(freq)
    use_median : bool
        Si True usa mediana en vez de media (más robusto)
    smooth_sigma : float
        Sigma del suavizado gaussiano
    """

    def compute_log_binned_stats(freqs, values, n_bins, use_median):
        log_freqs = np.log10(freqs)

        bins = np.linspace(log_freqs.min(), log_freqs.max(), n_bins + 1)
        bin_centers = 0.5 * (bins[:-1] + bins[1:])

        digitized = np.digitize(log_freqs, bins) - 1

        binned_vals = []
        valid_centers = []

        for i in range(n_bins):
            mask = digitized == i
            if np.sum(mask) > 0:
                if use_median:
                    agg = np.median(values[mask])
                else:
                    agg = np.mean(values[mask])

                binned_vals.append(agg)
                valid_centers.append(bin_centers[i])

        return 10**np.array(valid_centers), np.array(binned_vals)

    # ======================
    # Preparación de datos
    # ======================
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    classes_in_test, counts_test = np.unique(y_true, return_counts=True)

    recalls = []
    precisions = []
    freqs = []

    for cls, n_test in zip(classes_in_test, counts_test):
        if n_test < min_samples:
            continue

        r = recall_score(y_true == cls, y_pred == cls, zero_division=0)
        p = precision_score(y_true == cls, y_pred == cls, zero_division=0)

        freq = class_frequencies[cls] if isinstance(class_frequencies, dict) else class_frequencies[cls]

        recalls.append(r)
        precisions.append(p)
        freqs.append(freq)

    recalls = np.array(recalls)
    precisions = np.array(precisions)
    freqs = np.array(freqs)

    # Ordenar por frecuencia
    order = np.argsort(freqs)
    freqs = freqs[order]
    recalls = recalls[order]
    precisions = precisions[order]

    # ======================
    # Plot
    # ======================
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize, sharex=True)

    # ======================
    # Recall
    # ======================
    ax1.scatter(freqs, recalls, alpha=0.25, s=10)

    b_freqs, b_recalls = compute_log_binned_stats(freqs, recalls, n_bins, use_median)

    if len(b_recalls) > 5:
        smooth = gaussian_filter1d(b_recalls, sigma=smooth_sigma)
        ax1.plot(b_freqs, smooth, linestyle="--", linewidth=2, label="Gaussian trend")

    ax1.set_ylabel("Recall (per class)")
    ax1.set_title("Recall vs Class Frequency")
    ax1.set_ylim(-0.05, 1.05)
    ax1.grid(True, which="both", linestyle="--", alpha=0.3)
    ax1.legend()

    # ======================
    # Precision
    # ======================
    ax2.scatter(freqs, precisions, alpha=0.25, s=10)

    b_freqs, b_precisions = compute_log_binned_stats(freqs, precisions, n_bins, use_median)
    if len(b_precisions) > 5:
        smooth = gaussian_filter1d(b_precisions, sigma=smooth_sigma)
        ax2.plot(b_freqs, smooth, linestyle="--", linewidth=2, label="Gaussian trend")

    ax2.set_ylabel("Precision (per class)")
    ax2.set_xlabel("Training frequency (log scale)")
    ax2.set_title("Precision vs Class Frequency")
    ax2.set_ylim(-0.05, 1.05)
    ax2.grid(True, which="both", linestyle="--", alpha=0.3)
    ax2.legend()

    # Escala log
    ax2.set_xscale("log")

    plt.tight_layout()
    plt.show()

    return fig, (ax1, ax2)

In [ ]:
def plot_metric_diff_vs_frequency(
    y_true,
    y_pred1,
    y_pred2,
    class_frequencies,
    model_names=["Modelo 1", "Modelo 2"],
    min_samples=1,
    n_bins=20,
    use_median=False,
    smooth_sigma=1,
    show_raw_scatter=True,
    figsize=(14, 10),
):
    """
    Diferencia (M2 - M1) de Recall y Precision vs frecuencia con:
    - scatter raw (opcional)
    - scatter binned (promediado)
    - tendencia suavizada correcta en espacio log
    """

    def compute_log_binned_stats(freqs, values, n_bins, use_median):
        log_freqs = np.log10(freqs)

        bins = np.linspace(log_freqs.min(), log_freqs.max(), n_bins + 1)
        bin_centers = 0.5 * (bins[:-1] + bins[1:])
        digitized = np.digitize(log_freqs, bins) - 1

        binned_vals = []
        valid_centers = []
        counts = []

        for i in range(n_bins):
            mask = digitized == i
            if np.sum(mask) > 0:
                agg = np.median(values[mask]) if use_median else np.mean(values[mask])
                binned_vals.append(agg)
                valid_centers.append(bin_centers[i])
                counts.append(np.sum(mask))

        return (
            10**np.array(valid_centers),
            np.array(binned_vals),
            np.array(counts),
        )

    # ======================
    # Preparación
    # ======================
    y_true = np.asarray(y_true)
    y_pred1 = np.asarray(y_pred1)
    y_pred2 = np.asarray(y_pred2)

    classes_in_test, counts_test = np.unique(y_true, return_counts=True)

    diff_recalls, diff_precisions, freqs = [], [], []

    for cls, n_test in zip(classes_in_test, counts_test):
        if n_test < min_samples:
            continue

        r1 = recall_score(y_true == cls, y_pred1 == cls, zero_division=0)
        r2 = recall_score(y_true == cls, y_pred2 == cls, zero_division=0)

        p1 = precision_score(y_true == cls, y_pred1 == cls, zero_division=0)
        p2 = precision_score(y_true == cls, y_pred2 == cls, zero_division=0)

        freq = class_frequencies[cls]

        diff_recalls.append(r2 - r1)
        diff_precisions.append(p2 - p1)
        freqs.append(freq)

    freqs = np.array(freqs)
    diff_recalls = np.array(diff_recalls)
    diff_precisions = np.array(diff_precisions)

    order = np.argsort(freqs)
    freqs = freqs[order]
    diff_recalls = diff_recalls[order]
    diff_precisions = diff_precisions[order]

    # ======================
    # Binning
    # ======================
    b_freqs_r, b_recalls, counts_r = compute_log_binned_stats(freqs, diff_recalls, n_bins, use_median)
    b_freqs_p, b_precisions, counts_p = compute_log_binned_stats(freqs, diff_precisions, n_bins, use_median)

    # ======================
    # Plot
    # ======================
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize, sharex=True)

    metrics_data = [diff_recalls, diff_precisions]
    titles = [r"$\Delta$ Recall", r"$\Delta$ Precision"]
    colors = ["#1f77b4", "#ff7f0e"]

    for ax, raw_data, b_freqs, b_data, counts, title, color in zip(
        [ax1, ax2],
        metrics_data,
        [b_freqs_r, b_freqs_p],
        [b_recalls, b_precisions],
        [counts_r, counts_p],
        titles,
        colors,
    ):
        ax.axhline(0, color='black', linestyle='-', linewidth=1.2, alpha=0.5)

        # Scatter raw
        if show_raw_scatter:
            ax.scatter(freqs, raw_data, alpha=0.25, color=color, s=10)


        # Suavizado correcto
        if len(b_data) > 5:
            smooth = gaussian_filter1d(b_data, sigma=smooth_sigma)
            ax.plot(
                b_freqs,
                smooth,
                linestyle="--",
                linewidth=2,
                marker=None,
                color="red",
                label="Gaussian trend"
            )

        ax.set_ylabel(f"{title}\n({model_names[1]} - {model_names[0]})")
        ax.set_title(f"Difference in {title} vs Training Frequency", fontweight='bold')
        ax.set_ylim(-1.1, 1.1)
        ax.grid(True, which="both", linestyle="--", alpha=0.3)
        ax.legend(loc="upper right", fontsize='small')

    ax2.set_xscale("log")
    ax2.set_xlabel("Training frequency (log scale)")

    plt.tight_layout()
    plt.show()

    return fig, (ax1, ax2)

In [ ]:
def apply_simpleshot(train_feats, test_feats, test_labels, k=1, seed=42):
    rng = np.random.default_rng(seed)
    
    mean_train = np.mean(train_feats, axis=0)

    def preprocess(x):
        x = x - mean_train
        norm = np.linalg.norm(x, axis=1, keepdims=True)
        return x / (norm + 1e-10)

    train_feats_proc = preprocess(train_feats)
    test_feats_proc = preprocess(test_feats)

    unique_classes = np.unique(test_labels)
    prototypes, proto_labels = [], []
    mask_prototypes = np.zeros(len(test_labels), dtype=bool)
    
    for cls in unique_classes:
        cls_indices = np.where(test_labels == cls)[0]
        if len(cls_indices) > 0:
            n_take = min(len(cls_indices), k)
            selected = rng.choice(cls_indices, size=n_take, replace=False)
            mask_prototypes[selected] = True
            
            proto = np.mean(test_feats_proc[selected], axis=0)
            prototypes.append(proto)
            proto_labels.append(cls)

    X_proto, y_proto = np.array(prototypes), np.array(proto_labels)
    X_query, y_query = test_feats_proc[~mask_prototypes], test_labels[~mask_prototypes]

    knn = KNeighborsClassifier(n_neighbors=1, metric='euclidean', n_jobs=32)
    knn.fit(X_proto, y_proto)
    y_pred = knn.predict(X_query)

    return y_query, y_pred

In [ ]:
def benchmark_simpleshot(train_paths, k=1, seeds=[0, 7, 25, 42, 100]):
    """
    Realiza el benchmark de SimpleShot para múltiples experimentos y promedia resultados.
    
    Args:
        train_paths (dict): Diccionario {nombre_exp: path_a_embeddings_train}.
        k (int): Número de ejemplos por clase para formar los prototipos.
        seeds (list): Lista de semillas para promediar el sampleo.
    """
    taxa_levels = ['kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species']
    
    # Encabezado de la tabla
    header = f"{'Experimento':<40} | " + " | ".join([f"{t.capitalize():<6}" for t in taxa_levels])
    print(f"\nResultados SimpleShot (k={k}, runs={len(seeds)})")
    print(header)
    print("-" * len(header))

    for exp_name, train_path in train_paths.items():
        # Construir path de test basado en el nombre del experimento
        test_path = f"/lustre/fswork/projects/rech/tec/uod68bo/am/open_clip/tutorials/{exp_name}_test_results.npz"
        
        # Carga de datos
        try:
            train_feats = np.load(train_path)
            test_data = np.load(test_path)
        except FileNotFoundError:
            print(f"Error: No se encontraron archivos para {exp_name}")
            continue

        y_test_true = test_data['y_true']
        test_feats = test_data['features']

        accumulated_f1 = {t: [] for t in taxa_levels}
        # Ejecución de los runs con distintas semillas
        for s in seeds:
            y_true_ss, y_pred_ss = apply_simpleshot(
                train_feats, test_feats, y_test_true, k=k, seed=s
            )
            
            # 1. Métricas Taxonómicas (promediando por run)
            r = evaluate_taxonomic_metrics(y_true_ss, y_pred_ss, test_dataset.features["label"].names)
            for t in taxa_levels:
                accumulated_f1[t].append(r.get(t, {}).get('f1', 0.0))
            
        # Cálculo de promedios para la fila
        row_values = []
        for t in taxa_levels:
            avg_taxa_f1 = np.mean(accumulated_f1[t])
            row_values.append(f"{avg_taxa_f1:<6.3f}")
        
        
        # Imprimir fila del experimento
        print(f"{exp_name[:40]:<40} | " + " | ".join(row_values))

In [ ]:
def hierarchical_tsne_plot(
    dataset,
    features,
    hierarchy_dict,
    levels_order=("kingdom", "phylum", "class", "order", "family", "genus", "species"),
    perplexity=5,
    n_jobs=96,
    figsize=(20, 10)
):

    def parse_taxonomy(label):
        parts = label.split()
        parts = parts + [None] * (7 - len(parts))
        return parts[:7]

    taxonomy = pd.DataFrame(
        [parse_taxonomy(l) for l in dataset.features["label"].names],
        columns=levels_order
    )

    y_true = np.array(dataset["label"])

    df = pd.DataFrame({"y_true": y_true})
    df = df.join(taxonomy, on="y_true")

    selected_levels = [lvl for lvl in levels_order if lvl in hierarchy_dict]

    root_level = selected_levels[0]
    root_target = hierarchy_dict[root_level]

    mask_root = df[root_level] == root_target

    X_root = features[mask_root]
    df_root = df[mask_root].reset_index(drop=True)

    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        n_jobs=n_jobs,
        initialization="pca"
    )

    X_tsne = tsne.fit(X_root)

    x_min, x_max = X_tsne[:, 0].min() - 5, X_tsne[:, 0].max() + 5
    y_min, y_max = X_tsne[:, 1].min() - 5, X_tsne[:, 1].max() + 5

    n_plots = len(selected_levels)
    n_cols = min(3, n_plots)
    n_rows = int(np.ceil(n_plots / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)

    if n_plots == 1:
        axes = np.array([axes])
    else:
        axes = np.array(axes).reshape(-1)

    cmap = plt.get_cmap("tab20")

    current_mask = np.ones(len(df_root), dtype=bool)

    max_legend_rows = 1

    for i, level in enumerate(selected_levels):
        ax = axes[i]

        target_value = hierarchy_dict[level]
        current_mask = current_mask & (df_root[level] == target_value)

        df_plot = df_root[current_mask]
        X_plot = X_tsne[current_mask]

        level_idx = levels_order.index(level)
        if level_idx + 1 < len(levels_order):
            child_level = levels_order[level_idx + 1]
        else:
            child_level = level

        categories = df_plot[child_level].dropna().unique()

        for j, cat in enumerate(categories):
            idx = df_plot[child_level] == cat
            ax.scatter(
                X_plot[idx, 0],
                X_plot[idx, 1],
                s=5,
                alpha=0.7,
                color=cmap(j % 20),
                label=cat
            )

        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)

        ax.set_xticks([])
        ax.set_yticks([])

        ax.set_title(f"{child_level.capitalize()} of {target_value.capitalize()}", fontsize=16)

        # --- LEYENDA ABAJO (FIXED) ---
        n_cats = len(categories)
        if n_cats > 0:
            n_cols_legend = min(3, n_cats)

            # calcular filas de leyenda
            n_rows_legend = int(np.ceil(n_cats / n_cols_legend))
            max_legend_rows = max(max_legend_rows, n_rows_legend)

            ax.legend(
                markerscale=2,
                loc="upper center",
                bbox_to_anchor=(0, -0.1, 1, 0),  
                ncol=min(3, n_cats),
                fontsize="small",
                frameon=True
            )

    for j in range(len(selected_levels), len(axes)):
        axes[j].axis("off")

    fig.subplots_adjust(
        hspace=0.15,  
        wspace=0.1,    
        bottom=0.1 + 0.05 * max_legend_rows
    )

    plt.show()

In [ ]:
#import open_clip
#import torch

#model, _, _ = open_clip.create_model_and_transforms('')

#checkpoint_path = "/lustre/fswork/projects/rech/tec/uod68bo/am/open_clip/logs/2026_02_27-06_51_03-model_hf-hub:imageomics-bioclip-lr_0.0001-b_512-j_4-p_amp/checkpoints/epoch_95.pt"
#checkpoint_save = "/lustre/fswork/projects/rech/tec/uod68bo/am/open_clip/logs/2026_02_27-06_51_03-model_hf-hub:imageomics-bioclip-lr_0.0001-b_512-j_4-p_amp/checkpoints/epoch_95_fixed.pt"
#device = "cpu"

#checkpoint = torch.load(checkpoint_path, map_location=device)

#state_dict = checkpoint['state_dict']

# SOLUCIÓN DE PROBLEMAS COMUNES (DataParallel):
# Si entrenaste con múltiples GPUs, las keys pueden tener el prefijo 'module.'
#new_state_dict = {}
#for key, value in state_dict.items():
#    if key.startswith('module.'):
#        new_state_dict[key[7:]] = value
#    else:
#        new_state_dict[key] = value

# Cargar los pesos al modelo
#msg = model.load_state_dict(new_state_dict, strict=True)
#print(f"Pesos cargados. Reporte: {msg}")

#checkpoint['state_dict'] = model.cpu().state_dict()
#torch.save(checkpoint, checkpoint_save)

## Supervised

### ViT-B-16

In [ ]:
# ldam sin weights [listo]
# max_margin [listo]
# ral []

In [ ]:
ckpt_path = "/lustre/fswork/projects/rech/tec/uod68bo/am/planktonzilla/logs/train/multiruns/2026-03-19_05-16-57_1175096/2/checkpoint-470594/model.safetensors"

model = ClipClassifier(
    name="ViT-B-16",
    pretrained="",
    repo_path=None,
    num_features=768,
    num_labels=385
)

state_dict = load_file(ckpt_path)
model.load_state_dict(state_dict)
model = model.eval()

In [ ]:
transform = T.Compose([
        T.ToTensor(),
        T.Resize((224, 224), antialias=True),
        T.Normalize(mean=[0.481, 0.458, 0.408],
                    std=[0.269, 0.261, 0.276]),
    ])

y_true, y_pred, features_test = clf_preds_and_features(model, test_dataset, transform)

In [ ]:
np.savez("/lustre/fswork/projects/rech/tec/uod68bo/am/open_clip/tutorials/ViT-B-16-CLF_ral.npz", y_true=y_true, y_pred=y_pred, features=features_test)

# data = np.load(
#     "/lustre/fswork/projects/rech/tec/uod68bo/am/open_clip/tutorials/ViT-B-16-CLF.npz"
# )

# y_true = data["y_true"]
# y_pred = data["y_pred"]
# features = data["features"]

In [ ]:
r = evaluate_taxonomic_metrics(y_true, y_pred, test_dataset.features["label"].names)
r

In [ ]:
def plot_metrics_vs_frequency_multi(
    y_true,
    y_preds,
    names,
    class_frequencies,
    min_samples=1,
    n_bins=20,
    use_median=False,
    smooth_sigma=1,
    figsize=(14, 10),
):

    assert len(y_preds) == len(names), "y_preds y names deben tener mismo largo"

    def compute_log_binned_stats(freqs, values, n_bins, use_median):
        log_freqs = np.log10(freqs)

        bins = np.linspace(log_freqs.min(), log_freqs.max(), n_bins + 1)
        bin_centers = 0.5 * (bins[:-1] + bins[1:])

        digitized = np.digitize(log_freqs, bins) - 1

        binned_vals = []
        valid_centers = []

        for i in range(n_bins):
            mask = digitized == i
            if np.sum(mask) > 0:
                agg = np.median(values[mask]) if use_median else np.mean(values[mask])
                binned_vals.append(agg)
                valid_centers.append(bin_centers[i])

        return 10**np.array(valid_centers), np.array(binned_vals)

    # ======================
    # Preparación base
    # ======================
    y_true = np.asarray(y_true)
    classes_in_test, counts_test = np.unique(y_true, return_counts=True)

    valid_classes = []
    freqs = []

    for cls, n_test in zip(classes_in_test, counts_test):
        if n_test < min_samples:
            continue
        freq = class_frequencies[cls]
        valid_classes.append(cls)
        freqs.append(freq)

    freqs = np.array(freqs)
    order = np.argsort(freqs)
    freqs = freqs[order]
    valid_classes = np.array(valid_classes)[order]

    # ======================
    # Plot
    # ======================
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize, sharex=True)

    for y_pred, name in zip(y_preds, names):
        y_pred = np.asarray(y_pred)

        recalls = []
        precisions = []

        for cls in valid_classes:
            r = recall_score(y_true == cls, y_pred == cls, zero_division=0)
            p = precision_score(y_true == cls, y_pred == cls, zero_division=0)
            recalls.append(r)
            precisions.append(p)

        recalls = np.array(recalls)
        precisions = np.array(precisions)

        # ===== Recall =====
        b_freqs, b_recalls = compute_log_binned_stats(freqs, recalls, n_bins, use_median)
        if len(b_recalls) > 5:
            smooth = gaussian_filter1d(b_recalls, sigma=smooth_sigma)
            ax1.plot(b_freqs, smooth, linewidth=2, label=name)

        # ===== Precision =====
        b_freqs, b_precisions = compute_log_binned_stats(freqs, precisions, n_bins, use_median)
        if len(b_precisions) > 5:
            smooth = gaussian_filter1d(b_precisions, sigma=smooth_sigma)
            ax2.plot(b_freqs, smooth, linewidth=2, label=name)

    # ======================
    # Estética
    # ======================
    ax1.set_ylabel("Recall (per class)")
    ax1.set_title("Recall vs Class Frequency")
    ax1.set_ylim(-0.05, 1.05)
    ax1.grid(True, which="both", linestyle="--", alpha=0.3)
    ax1.legend()

    ax2.set_ylabel("Precision (per class)")
    ax2.set_xlabel("Training frequency (log scale)")
    ax2.set_title("Precision vs Class Frequency")
    ax2.set_ylim(-0.05, 1.05)
    ax2.grid(True, which="both", linestyle="--", alpha=0.3)
    ax2.legend()

    ax2.set_xscale("log")

    plt.tight_layout()
    plt.show()

    return fig, (ax1, ax2)

In [ ]:
data = np.load(
    "/lustre/fswork/projects/rech/tec/uod68bo/am/open_clip/tutorials/ViT-B-16-CLF.npz"
)

y_true = data["y_true"]
y_pred = data["y_pred"]

data = np.load(
    "/lustre/fswork/projects/rech/tec/uod68bo/am/open_clip/tutorials/ViT-B-16-CLF_max_margin.npz"
)

y_pred2 = data["y_pred"]

data = np.load(
    "/lustre/fswork/projects/rech/tec/uod68bo/am/open_clip/tutorials/ViT-B-16-CLF_ldam_sin_weights.npz"
)

y_pred3 = data["y_pred"]

data = np.load(
    "/lustre/fswork/projects/rech/tec/uod68bo/am/open_clip/tutorials/ViT-B-16-CLF_ral.npz"
)

y_pred4 = data["y_pred"]

In [ ]:
def plot_metrics_vs_frequency_multi(
    y_true,
    y_preds,
    names,
    class_frequencies,
    cmap_name="viridis",
    min_samples=1,
    n_bins=20,
    use_median=False,
    smooth_sigma=1,
    figsize=(14, 10),
):
    import numpy as np
    import matplotlib.pyplot as plt
    from sklearn.metrics import recall_score, precision_score
    from scipy.ndimage import gaussian_filter1d
    from matplotlib import cm
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

    assert len(y_preds) == len(names), "y_preds y names deben tener mismo largo"

    cmap = cm.get_cmap(cmap_name, len(names))
    colors = [cmap(i) for i in range(len(names))]

    def compute_log_binned_stats(freqs, values):
        log_freqs = np.log10(freqs)
        bins = np.linspace(log_freqs.min(), log_freqs.max(), n_bins + 1)
        bin_centers = 0.5 * (bins[:-1] + bins[1:])
        digitized = np.digitize(log_freqs, bins) - 1

        binned_vals, valid_centers = [], []

        for i in range(n_bins):
            mask = digitized == i
            if np.sum(mask) > 0:
                agg = np.median(values[mask]) if use_median else np.mean(values[mask])
                binned_vals.append(agg)
                valid_centers.append(bin_centers[i])

        return 10**np.array(valid_centers), np.array(binned_vals)

    def add_background_hist(ax, freqs):
        ax_hist = ax.twinx()

        # Hist relleno
        ax_hist.hist(
            freqs,
            bins=50,
            color="gray",
            alpha=0.5,
            histtype="stepfilled"
        )

        # Bordes oscuros
        ax_hist.hist(
            freqs,
            bins=50,
            histtype="step",
            color="black",
            linewidth=1
        )

        ax_hist.set_yticks([])
        ax_hist.set_ylabel("")

        # Mandar al fondo correctamente
        ax_hist.set_zorder(0)
        ax.set_zorder(1)
        ax.patch.set_alpha(0)

        return ax_hist

    def add_zoom(ax):
        x_min = 1e3
        x_max = freqs.max()

        axins = inset_axes(ax, width="40%", height="40%", loc="upper left")

        for line in ax.get_lines():
            x, y = line.get_data()
            mask = x >= x_min
            if np.sum(mask) > 0:
                axins.plot(x[mask], y[mask], linewidth=2)

        axins.set_xscale("log")
        axins.set_xlim(x_min, x_max)
        axins.set_ylim(0, 1)

        # Caja + "flecha visual"
        mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="black")

        return axins

    # ======================
    # Preparación datos
    # ======================
    y_true = np.asarray(y_true)
    classes, counts = np.unique(y_true, return_counts=True)

    valid_classes, freqs = [], []

    for cls, n in zip(classes, counts):
        if n < min_samples:
            continue
        valid_classes.append(cls)
        freqs.append(class_frequencies[cls])

    freqs = np.array(freqs)
    order = np.argsort(freqs)

    freqs = freqs[order]
    valid_classes = np.array(valid_classes)[order]

    # ======================
    # Plot base
    # ======================
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize, sharex=True)

    # 👉 Histogramas en ambos plots
    add_background_hist(ax1, freqs)
    add_background_hist(ax2, freqs)

    # ======================
    # Curvas por modelo
    # ======================
    for y_pred, name, color in zip(y_preds, names, colors):
        y_pred = np.asarray(y_pred)

        recalls, precisions = [], []

        for cls in valid_classes:
            recalls.append(recall_score(y_true == cls, y_pred == cls, zero_division=0))
            precisions.append(precision_score(y_true == cls, y_pred == cls, zero_division=0))

        recalls = np.array(recalls)
        precisions = np.array(precisions)

        # Recall
        b_f, b_r = compute_log_binned_stats(freqs, recalls)
        if len(b_r) > 5:
            smooth = gaussian_filter1d(b_r, sigma=smooth_sigma)
            ax1.plot(b_f, smooth, color=color, linewidth=2, label=name)

        # Precision
        b_f, b_p = compute_log_binned_stats(freqs, precisions)
        if len(b_p) > 5:
            smooth = gaussian_filter1d(b_p, sigma=smooth_sigma)
            ax2.plot(b_f, smooth, color=color, linewidth=2, label=name)

    # 👉 Zoom en ambos
    add_zoom(ax1)
    add_zoom(ax2)

    # ======================
    # Estética
    # ======================
    ax1.set_ylabel("Recall")
    ax1.set_ylim(-0.05, 1.05)
    ax1.set_title("Recall vs Class Frequency")
    ax1.grid(True, linestyle="--", alpha=0.3)
    ax1.legend()

    ax2.set_ylabel("Precision")
    ax2.set_xlabel("Training frequency (log scale)")
    ax2.set_ylim(-0.05, 1.05)
    ax2.set_title("Precision vs Class Frequency")
    ax2.grid(True, linestyle="--", alpha=0.3)
    ax2.legend()

    ax2.set_xscale("log")

    plt.tight_layout()
    plt.show()

    return fig, (ax1, ax2)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from sklearn.metrics import recall_score, precision_score

def plot_metrics_vs_frequency_multi2(
    y_true,
    y_preds,
    names,
    class_frequencies,
    min_samples=1,
    n_bins=20,
    use_median=False,
    smooth_sigma=1,
    figsize=(14, 10),
    cmap_name="viridis"  # 1. Input para el colormap
):

    assert len(y_preds) == len(names), "y_preds y names deben tener mismo largo"

    def compute_log_binned_stats(freqs, values, n_bins, use_median):
        log_freqs = np.log10(freqs)

        bins = np.linspace(log_freqs.min(), log_freqs.max(), n_bins + 1)
        bin_centers = 0.5 * (bins[:-1] + bins[1:])

        digitized = np.digitize(log_freqs, bins) - 1

        binned_vals = []
        valid_centers = []

        for i in range(n_bins):
            mask = digitized == i
            if np.sum(mask) > 0:
                agg = np.median(values[mask]) if use_median else np.mean(values[mask])
                binned_vals.append(agg)
                valid_centers.append(bin_centers[i])

        return 10**np.array(valid_centers), np.array(binned_vals)

    # ======================
    # Preparación base
    # ======================
    y_true = np.asarray(y_true)
    classes_in_test, counts_test = np.unique(y_true, return_counts=True)

    valid_classes = []
    freqs = []

    for cls, n_test in zip(classes_in_test, counts_test):
        if n_test < min_samples:
            continue
        freq = class_frequencies[cls]
        valid_classes.append(cls)
        freqs.append(freq)

    freqs = np.array(freqs)
    order = np.argsort(freqs)
    freqs = freqs[order]
    valid_classes = np.array(valid_classes)[order]

    # ======================
    # Preparación Colores y Plot
    # ======================
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize, sharex=True)
    
    # Obtener colores del colormap seleccionado
    cmap = plt.get_cmap(cmap_name)
    colors = [cmap(i) for i in np.linspace(0, 1, len(y_preds))]

    # 2. Histograma de fondo
    # Bins logarítmicos para que se vean de igual ancho (lineales) en el eje x logarítmico
    hist_bins = np.logspace(np.log10(freqs.min()), np.log10(freqs.max()), n_bins + 1)
    
    for ax in [ax1, ax2]:
        ax_hist = ax.twinx()
        # Color plomo, alpha 0.5, stepfilled para rellenar pero mantener bordes fuertes
        ax_hist.hist(freqs, bins=hist_bins, facecolor='gray', alpha=0.5, 
                     edgecolor='#333333', linewidth=1.2, histtype='stepfilled')
        ax_hist.set_yticks([])  # Ocultar el eje Y
        
        # Ajuste de z-order para que las líneas de métricas pasen por encima del histograma
        ax.set_zorder(ax_hist.get_zorder() + 1)
        ax.patch.set_visible(False)

    # 3. Preparar Insets (Zoom)
    # Ubicamos las cajas en la parte inferior izquierda [x0, y0, width, height] (relativo al tamaño del plot)
    ax1_ins = ax1.inset_axes([0.05, 0.05, 0.45, 0.4])
    ax2_ins = ax2.inset_axes([0.05, 0.05, 0.45, 0.4])
    
    zoom_x_min = 1000  # Inicio del zoom
    # Variables para ajustar el límite Y de las cajas de zoom automáticamente
    ax1_y_min, ax1_y_max = 1.0, 0.0
    ax2_y_min, ax2_y_max = 1.0, 0.0

    for y_pred, name, color in zip(y_preds, names, colors):
        y_pred = np.asarray(y_pred)

        recalls = []
        precisions = []

        for cls in valid_classes:
            r = recall_score(y_true == cls, y_pred == cls, zero_division=0)
            p = precision_score(y_true == cls, y_pred == cls, zero_division=0)
            recalls.append(r)
            precisions.append(p)

        recalls = np.array(recalls)
        precisions = np.array(precisions)

        # ===== Recall =====
        b_freqs, b_recalls = compute_log_binned_stats(freqs, recalls, n_bins, use_median)
        if len(b_recalls) > 5:
            smooth = gaussian_filter1d(b_recalls, sigma=smooth_sigma)
            ax1.plot(b_freqs, smooth, linewidth=2, label=name, color=color)
            ax1_ins.plot(b_freqs, smooth, linewidth=2, color=color)  # Dibujar en el zoom
            
            # Registrar mínimos y máximos para la zona del zoom
            mask = b_freqs >= zoom_x_min
            if np.any(mask):
                ax1_y_min = min(ax1_y_min, smooth[mask].min())
                ax1_y_max = max(ax1_y_max, smooth[mask].max())

        # ===== Precision =====
        b_freqs, b_precisions = compute_log_binned_stats(freqs, precisions, n_bins, use_median)
        if len(b_precisions) > 5:
            smooth = gaussian_filter1d(b_precisions, sigma=smooth_sigma)
            ax2.plot(b_freqs, smooth, linewidth=2, label=name, color=color)
            ax2_ins.plot(b_freqs, smooth, linewidth=2, color=color) # Dibujar en el zoom
            
            # Registrar mínimos y máximos para la zona del zoom
            mask = b_freqs >= zoom_x_min
            if np.any(mask):
                ax2_y_min = min(ax2_y_min, smooth[mask].min())
                ax2_y_max = max(ax2_y_max, smooth[mask].max())

    # ======================
    # Configurar y conectar cajas de Zoom
    # ======================
    for ax_ins, y_min, y_max, main_ax in zip([ax1_ins, ax2_ins], 
                                             [ax1_y_min, ax2_y_min], 
                                             [ax1_y_max, ax2_y_max], 
                                             [ax1, ax2]):
        # Solo aplicar si hubo datos en ese rango
        if y_min <= y_max:
            margin = max(0.01, (y_max - y_min) * 0.15)
            ax_ins.set_xlim(zoom_x_min, freqs.max())
            ax_ins.set_ylim(y_min - margin, y_max + margin)
            ax_ins.set_xscale("log")
            ax_ins.grid(True, which="both", linestyle="--", alpha=0.3)
            # Crea la caja sobre el >10^3 y traza las líneas conectándolo con el inset
            main_ax.indicate_inset_zoom(ax_ins, edgecolor="black", alpha=0.6)

    # ======================
    # Estética
    # ======================
    ax1.set_ylabel("Recall (per class)")
    ax1.set_title("Recall vs Class Frequency")
    ax1.set_ylim(-0.05, 1.05)
    ax1.grid(True, which="both", linestyle="--", alpha=0.3)
    ax1.legend()

    ax2.set_ylabel("Precision (per class)")
    ax2.set_xlabel("Training frequency (log scale)")
    ax2.set_title("Precision vs Class Frequency")
    ax2.set_ylim(-0.05, 1.05)
    ax2.grid(True, which="both", linestyle="--", alpha=0.3)
    ax2.legend()

    ax2.set_xscale("log")

    plt.tight_layout()
    plt.show()

    return fig, (ax1, ax2)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from sklearn.metrics import recall_score, precision_score

def plot_metrics_vs_frequency_multi(
    y_true,
    y_preds,
    names,
    class_frequencies,
    min_samples=1,
    n_bins=20,
    use_median=False,
    smooth_sigma=1,
    figsize=(14, 10),
    cmap_name="viridis"
):

    assert len(y_preds) == len(names), "y_preds y names deben tener mismo largo"

    def compute_log_binned_stats(freqs, values, n_bins, use_median):
        log_freqs = np.log10(freqs)

        bins = np.linspace(log_freqs.min(), log_freqs.max(), n_bins + 1)
        bin_centers = 0.5 * (bins[:-1] + bins[1:])

        digitized = np.digitize(log_freqs, bins) - 1

        binned_vals = []
        valid_centers = []

        for i in range(n_bins):
            mask = digitized == i
            if np.sum(mask) > 0:
                agg = np.median(values[mask]) if use_median else np.mean(values[mask])
                binned_vals.append(agg)
                valid_centers.append(bin_centers[i])

        return 10**np.array(valid_centers), np.array(binned_vals)

    # ======================
    # Preparación base
    # ======================
    y_true = np.asarray(y_true)
    classes_in_test, counts_test = np.unique(y_true, return_counts=True)

    valid_classes = []
    freqs = []

    for cls, n_test in zip(classes_in_test, counts_test):
        if n_test < min_samples:
            continue
        freq = class_frequencies[cls]
        valid_classes.append(cls)
        freqs.append(freq)

    freqs = np.array(freqs)
    order = np.argsort(freqs)
    freqs = freqs[order]
    valid_classes = np.array(valid_classes)[order]

    # ======================
    # Preparación Colores y Plot
    # ======================
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize, sharex=True)
    
    cmap = plt.get_cmap(cmap_name)
    colors = [cmap(i) for i in np.linspace(0, 1, len(y_preds))]

    hist_bins = np.logspace(np.log10(freqs.min()), np.log10(freqs.max()), n_bins + 1)
    
    for ax in [ax1, ax2]:
        ax_hist = ax.twinx()
        ax_hist.hist(freqs, bins=hist_bins, facecolor='gray', alpha=0.2, 
                     edgecolor='#333333', linewidth=1.2, histtype='stepfilled')
        ax_hist.set_yticks([]) 
        ax.set_zorder(ax_hist.get_zorder() + 1)
        ax.patch.set_visible(False)

    # ======================
    # 3. Cajas Insets (Zoom) ajustadas
    # [x0, y0, ancho, alto]
    # Movidas más arriba (0.15), más a la derecha (0.62) y más altas (0.55)
    # ======================
    ax1_ins = ax1.inset_axes([0.62, 0.15, 0.35, 0.55])
    ax2_ins = ax2.inset_axes([0.62, 0.15, 0.35, 0.55])
    
    zoom_x_min = 10**3
    zoom_x_max = 10**5
    
    ax1_y_min, ax1_y_max = 1.0, 0.0
    ax2_y_min, ax2_y_max = 1.0, 0.0

    for y_pred, name, color in zip(y_preds, names, colors):
        y_pred = np.asarray(y_pred)

        recalls = []
        precisions = []

        for cls in valid_classes:
            r = recall_score(y_true == cls, y_pred == cls, zero_division=0)
            p = precision_score(y_true == cls, y_pred == cls, zero_division=0)
            recalls.append(r)
            precisions.append(p)

        recalls = np.array(recalls)
        precisions = np.array(precisions)

        # ===== Recall =====
        b_freqs, b_recalls = compute_log_binned_stats(freqs, recalls, n_bins, use_median)
        if len(b_recalls) > 5:
            smooth = gaussian_filter1d(b_recalls, sigma=smooth_sigma)
            ax1.plot(b_freqs, smooth, linewidth=2, label=name, color=color)
            ax1_ins.plot(b_freqs, smooth, linewidth=2, color=color)
            
            # Restringimos el cálculo de y_min e y_max estrictamente al rango del zoom
            mask = (b_freqs >= zoom_x_min) & (b_freqs <= zoom_x_max)
            if np.any(mask):
                ax1_y_min = min(ax1_y_min, smooth[mask].min())
                ax1_y_max = max(ax1_y_max, smooth[mask].max())

        # ===== Precision =====
        b_freqs, b_precisions = compute_log_binned_stats(freqs, precisions, n_bins, use_median)
        if len(b_precisions) > 5:
            smooth = gaussian_filter1d(b_precisions, sigma=smooth_sigma)
            ax2.plot(b_freqs, smooth, linewidth=2, label=name, color=color)
            ax2_ins.plot(b_freqs, smooth, linewidth=2, color=color) 
            
            # Restringimos el cálculo de y_min e y_max estrictamente al rango del zoom
            mask = (b_freqs >= zoom_x_min) & (b_freqs <= zoom_x_max)
            if np.any(mask):
                ax2_y_min = min(ax2_y_min, smooth[mask].min())
                ax2_y_max = max(ax2_y_max, smooth[mask].max())

    # Conectamos las cajas con sus respectivos límites
    for ax_ins, y_min, y_max, main_ax in zip([ax1_ins, ax2_ins], 
                                             [ax1_y_min, ax2_y_min], 
                                             [ax1_y_max, ax2_y_max], 
                                             [ax1, ax2]):
        if y_min <= y_max:
            margin = max(0.005, (y_max - y_min) * 0.15)
            ax_ins.set_xlim(zoom_x_min, zoom_x_max)
            ax_ins.set_ylim(y_min - margin, y_max + margin)
            ax_ins.set_xscale("log")
            ax_ins.grid(True, which="both", linestyle="--", alpha=0.3)
            main_ax.indicate_inset_zoom(ax_ins, edgecolor="black", alpha=0.6)

    # ======================
    # Estética
    # ======================
    ax1.set_ylabel("Recall (per class)", fontsize=14)
    ax1.set_title("Recall vs Class Frequency", fontsize=16)
    ax1.set_ylim(-0.05, 1.05)
    ax1.grid(True, which="both", linestyle="--", alpha=0.3)
    ax1.legend(loc="upper left") 

    ax2.set_ylabel("Precision (per class)", fontsize=14)
    ax2.set_xlabel("Training frequency (log scale)", fontsize=14)
    ax2.set_title("Precision vs Class Frequency", fontsize=16)
    ax2.set_ylim(-0.05, 1.05)
    ax2.grid(True, which="both", linestyle="--", alpha=0.3)
    ax2.legend(loc="upper left") 

    ax2.set_xscale("log")

    plt.tight_layout()
    plt.show()

    return fig, (ax1, ax2)

In [ ]:
a,b = plot_metrics_vs_frequency_multi(
    y_true=y_true,
    y_preds = [y_pred, y_pred2, y_pred3, y_pred4],
    names = ["CE", "Max_margin", "LDAM sin weights", "RAL"],
    class_frequencies = class_frequencies,
    min_samples=1,
    n_bins=40,
    use_median=False,
    smooth_sigma=1,
    figsize=(14, 10),
)

In [ ]:
data = np.load(
    "/lustre/fswork/projects/rech/tec/uod68bo/am/open_clip/tutorials/ViT-B-16-CLF.npz"
)

features = data["features"]

In [ ]:
hierarchy_dict = {
    "kingdom": "chromista",
    "class": "bacillariophyceae",
    "genus": "chaetoceros"
}

hierarchical_tsne_plot(
    dataset=test_dataset,
    features=features,
    hierarchy_dict=hierarchy_dict,
    perplexity=10,
    n_jobs=96,
    figsize=(20, 10)
)

### ViT-L-14

In [ ]:
ckpt_path = "/lustre/fswork/projects/rech/tec/uod68bo/am/planktonzilla/logs/train/runs/2026-03-08_03-47-21_688363/checkpoint-304502/model.safetensors"

model = ClipClassifier(
    name="ViT-L-14",
    pretrained="",
    repo_path=None,
    num_features=1024,
    num_labels=385
)

state_dict = load_file(ckpt_path)
model.load_state_dict(state_dict)
model = model.eval()

In [ ]:
transform = T.Compose([
        T.ToTensor(),
        T.Resize((224, 224), antialias=True),
        T.Normalize(mean=[0.5, 0.5, 0.5],
                    std=[0.5, 0.5, 0.5]),
    ])

y_true, y_pred, features_test = clf_preds_and_features(model, test_dataset, transform)

In [ ]:
#np.savez("/lustre/fswork/projects/rech/tec/uod68bo/am/open_clip/tutorials/ViT-L-14-CLF.npz", y_true=y_true, y_pred=y_pred, features=features_test)

data = np.load(
    "/lustre/fswork/projects/rech/tec/uod68bo/am/open_clip/tutorials/ViT-L-14-CLF.npz"
)

y_true = data["y_true"]
y_pred = data["y_pred"]
#features = data["features"]

In [ ]:
#Macro Precision: 0.8941
#Macro Recall:    0.8646
#Macro F1:        0.8738
r = evaluate_taxonomic_metrics(y_true, y_pred, test_dataset.features["label"].names)
r

### EVA02-L-14

In [ ]:
ckpt_path = "/lustre/fswork/projects/rech/tec/uod68bo/am/planktonzilla/logs/train/runs/2026-03-08_04-15-25_688443/checkpoint-498276/model.safetensors"

model = ClipClassifier(
    name="EVA02-L-14",
    pretrained="",
    repo_path=None,
    num_features=1024,
    num_labels=385
)

state_dict = load_file(ckpt_path)
model.load_state_dict(state_dict)
model = model.eval()

In [ ]:
transform = T.Compose([
        T.ToTensor(),
        T.Resize((224, 224), antialias=True),
        T.Normalize(mean=[0.481, 0.458, 0.408],
                    std=[0.269, 0.261, 0.276]),
    ])

y_true, y_pred = infer(model, test_dataset, transform)

In [ ]:
np.savez("/lustre/fswork/projects/rech/tec/uod68bo/am/open_clip/tutorials/predicciones_eva02-l-14-clip_clf.npz", y_true=y_true, y_pred=y_pred)

data = np.load(
    "/lustre/fswork/projects/rech/tec/uod68bo/am/open_clip/tutorials/predicciones_eva02-l-14-clip_clf.npz"
)

y_true = data["y_true"]
y_pred = data["y_pred"]

In [ ]:
r = evaluate_taxonomic_metrics(y_true, y_pred, test_dataset.features["label"].names)
r

In [ ]:
plot_confusion_matrix_with_taxonomy(
    y_true=y_true,
    y_pred=y_pred,
    label_names_full=test_dataset.features["label"].names,
    taxonomy_levels=["kingdom", "phylum"],
    taxonomy_colors=["black", "purple"],
    #savepath="confusion_matrix_taxonomy.png",
)

## CLIP

In [ ]:
modelos = {
           "ViT-B-16 OpenAI-Planktonzilla": 'hf-hub:project-oceania/CLIP-ViT-B-16.openai-pt.planktonzilla-pt',
           "ViT-B-16 BioCLIP-Planktonzilla": 'hf-hub:project-oceania/CLIP-ViT-B-16.bioclip-pt.planktonzilla-pt',
           #"ViT-B-16 BioCLIP": 'hf-hub:imageomics/bioclip',
           #"ViT-L-14 Laion2b-Planktonzilla": 'hf-hub:project-oceania/CLIP-ViT-L-14.laion2b-pt.planktonzilla-pt',
           #"ViT-L-14 BioCLIP2-Planktonzilla": 'hf-hub:project-oceania/CLIP-ViT-L-14.bioclip2-pt.planktonzilla-pt',
           #"ViT-L-14 BioCLIP2": 'hf-hub:imageomics/bioclip-2',
          }

In [ ]:
class_names = test_dataset.features["label"].names

for exp_name, model_name in modelos.items():

    
    model, _, preprocess = open_clip.create_model_and_transforms(model_name)
    tokenizer = open_clip.get_tokenizer(model_name)

    _, _, all_features = clip_preds_and_features(
                                            model,
                                            train_dataset,
                                            tokenizer,
                                            preprocess,
                                            class_names,
                                            prompt_template=None
                                        )

    np.save(f"/lustre/fswork/projects/rech/tec/uod68bo/am/open_clip/tutorials/{exp_name}_train_features.npy", all_features)

In [ ]:
test_labels = np.array(test_dataset["label"])
test_unique_indices = np.unique(test_labels).tolist() 
class_names = test_dataset.features["label"].names

my_prompt = "a photo of a {label}"

for exp_name, model_name in modelos.items():
    model, _, preprocess = open_clip.create_model_and_transforms(model_name)
    tokenizer = open_clip.get_tokenizer(model_name)

    # Pasamos los índices únicos como máscara
    targets, preds, all_features = clip_preds_and_features(
        model,
        test_dataset, 
        tokenizer,
        preprocess,
        class_names, 
        prompt_template=my_prompt,
        allowed_indices=test_unique_indices 
    )

    output_path = f"/lustre/fswork/projects/rech/tec/uod68bo/am/open_clip/tutorials/{exp_name}_with_promp_test_results.npz"
    np.savez(output_path, features=all_features, y_true=targets, y_preds=preds)

In [ ]:
experimentos = {"ViT-B-16 OpenAI-Planktonzilla": 'ViT-B-16 OpenAI-Planktonzilla_with_promp_test_results.npz',
                           "ViT-B-16 BioCLIP-Planktonzilla": 'ViT-B-16 BioCLIP-Planktonzilla_with_promp_test_results.npz',
                           "ViT-B-16 BioCLIP": 'ViT-B-16 BioCLIP_with_promp_test_results.npz',
                           "ViT-L-14 Laion2b-Planktonzilla": 'ViT-L-14 Laion2b-Planktonzilla_with_promp_test_results.npz',
                           "ViT-L-14 BioCLIP2-Planktonzilla": 'ViT-L-14 BioCLIP2-Planktonzilla_with_promp_test_results.npz',
                           "ViT-L-14 BioCLIP2": 'ViT-L-14 BioCLIP2_with_promp_test_results.npz',
                          }

In [ ]:
taxa_levels = ['kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species']

# Encabezado de la tabla
header = f"{'Experimento':<40} | " + " | ".join([f"{t.capitalize():<8}" for t in taxa_levels]) + " | Original F1"
separator = "-" * len(header)

print(header)
print(separator)

for exp_name, results_path in experimentos.items():
    # Cargar datos
    data = np.load(results_path) # Usamos la variable results_path del loop
    y_true = data["y_true"]
    y_pred = data["y_preds"]
    
    f1 = f1_score(y_true, y_pred,
                average="macro", zero_division=0)

    # Calcular métricas taxonómicas
    r = evaluate_taxonomic_metrics(y_true, y_pred, test_dataset.features["label"].names)
    
    # Extraer f1 de cada nivel 
    row_values = []
    for t in taxa_levels:
        val = r.get(t, {}).get('f1', 0.0)
        row_values.append(f"{val:<8.3f}")
    
    
    # Imprimir fila formateada
    print(f"{exp_name[:40]:<40} | " + " | ".join(row_values) + f" | {f1:.3f}")

In [ ]:
train_paths = {
               "ViT-B-16 BioCLIP": 'ViT-B-16 BioCLIP_train_features.npy',
               "ViT-L-14 BioCLIP2": 'ViT-L-14 BioCLIP2_train_features.npy',
              }

In [ ]:
benchmark_simpleshot(train_paths, k=1)


In [ ]:
benchmark_simpleshot(train_paths, k=5)

In [ ]:
benchmark_simpleshot(train_paths, k=10)

In [ ]:
data = np.load(
    "ViT-B-16 OpenAI-Planktonzilla_test_results.npz"
)

y_true = data["y_true"]
y_pred = data["y_preds"]
features = data["features"]

In [ ]:
data = np.load(
    "ViT-B-16 OpenAI-Planktonzilla_test_results.npz"
)

y_true = data["y_true"]
y_pred = data["y_preds"]

data = np.load(
    "/lustre/fswork/projects/rech/tec/uod68bo/am/open_clip/tutorials/ViT-B-16-CLF.npz"
)

y_pred2 = data["y_pred"]

In [ ]:
plot_metrics_vs_frequency(
    y_true,
    y_pred,
    class_frequencies,
    min_samples=1,
    n_bins=40,
    use_median=False,
    smooth_sigma=1,
    figsize=(14, 10),
)

In [ ]:
plot_metric_diff_vs_frequency(
    y_true,
    y_pred,
    y_pred2,
    class_frequencies,
    model_names=["CLIP", "Clasificador"],
    min_samples=1,
    n_bins=50,
    smooth_sigma=1,
    figsize=(14, 10),
)